In [ ]:
from requirements_deprecated import *

# 1. Active Space

In [ ]:
filename_aer = "co2_active_space_comparison_Aer_2468_final.pkl"
raw_data_aer = load_data(filename_aer) 

filename_classical = "co2_classical_benchmarks.pkl"
raw_data_classical = load_data(filename_classical)

records = []

if raw_data_aer is not None:
    for cas_label, data_list in raw_data_aer.items():
        for data_point in data_list:
            dist = data_point["distance"]
            energy = data_point["energy"]
            metrics = data_point.get("metrics", {})
            
            evals = metrics.get('cost_function_evals', metrics.get('nfev', 0))
            iterations = metrics.get('nit', 0)
            total_time = metrics.get('total_time', 0.0)
            
            records.append({
                "Distance": dist,
                "Method": "aer_statevector", 
                "Active_Space": cas_label,
                "Energy": energy,
                "Time": total_time,
                "Evaluations": evals,
                "Iterations": iterations
            })

if raw_data_classical is not None:
    for data_point in raw_data_classical:
        dist = data_point["distance"]
        
        if data_point.get("energy_fci") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "FCI", 
                            "Energy": data_point["energy_fci"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})
        
        if data_point.get("energy_hf") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "HF", 
                            "Energy": data_point["energy_hf"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})
        
        if data_point.get("energy_dft") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "DFT", 
                            "Energy": data_point["energy_dft"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})

df_co2 = pd.DataFrame(records)

In [ ]:
def plot_active_space_analysis(df, method_filter, config, visuals, use_visuals=False, save_image=False):
    subset = df[(df["Method"] == method_filter) | (df["Method"] == "classical")].copy()
    if subset.empty: return
    
    classical_refs = ["FCI", "HF", "DFT"]
    active_spaces = [a for a in subset["Active_Space"].unique() if a not in classical_refs]
    
    include_fci = config.get("include_fci_baseline", True)
    other_baselines = config.get("include_other_baselines", [])
    
    palette = sns.color_palette("tab10", n_colors=len(active_spaces))
    color_map = dict(zip(active_spaces, palette))
    
    color_map["FCI"] = "#000000"    
    color_map["HF"] = "#9467bd"     
    color_map["DFT"] = "#17becf"    
    
    base_marker_map = dict(zip(active_spaces, config["markers"][:len(active_spaces)]))
    for ref in classical_refs:
        base_marker_map[ref] = ""

    panels = config.get("panels", [])
    ncols = len(panels)
    if ncols == 0: return
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 5)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h))
    
    if ncols == 1: axes = [axes]
        
    plt.subplots_adjust(wspace=config["wspace"])

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 1.16, min(y)], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.5))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    fci_min_x, fci_min_y = None, None
    if include_fci and "FCI" in subset["Active_Space"].values:
        fci_data_df = subset[subset["Active_Space"] == "FCI"].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values
        if len(fci_dist) > 0:
            zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    legend_handles = []

    for idx, panel in enumerate(panels):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        y_col = "Energy"
        if panel == "time": y_col = "Time"
        elif panel == "iterations": y_col = "Iterations"
        elif panel == "evaluations": y_col = "Evaluations"
        
        opts_to_plot = []
        if panel in ["energy_raw", "energy_zoomed"]:
            if include_fci and "FCI" in subset["Active_Space"].values:
                opts_to_plot.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Active_Space"].values]
            opts_to_plot.extend(valid_others)
        opts_to_plot.extend(active_spaces)
        
        groups = p_cfg.get("groups", [])
        if not groups: groups = [active_spaces] 
        
        for b_idx, space_name in enumerate(opts_to_plot):
            d = subset[subset["Active_Space"] == space_name].sort_values("Distance").dropna(subset=[y_col])
            
            if panel == "time" and config.get("log_time", True):
                d = d[d[y_col] > 0]
                
            if len(d) == 0: continue
            
            c = color_map.get(space_name, "grey")
            mk = base_marker_map.get(space_name, "o")
            
            ls_dict = p_cfg.get("line_styles")
            ls = ls_dict.get(space_name, "-" if space_name not in classical_refs else "--") if ls_dict else ("-" if space_name not in classical_refs else "--")
            show_mk = p_cfg.get("show_markers", True) and space_name not in classical_refs
            
            show_outline = p_cfg.get("show_outline", False)
            edge_c = config.get("outline_color", "white") if show_outline else "none"
            line_w = config.get("outline_width", 1.0) if show_outline else 0.0
            
            ms = config.get("marker_size", 50)
            mh_dict = p_cfg.get("marker_hierarchy")
            if mh_dict is not None: ms = mh_dict.get(space_name, ms)
            if panel == "energy_zoomed": ms *= 1.5
            
            alpha_val = config.get("alpha", 0.85)
            ah_dict = p_cfg.get("alpha_hierarchy")
            if ah_dict is not None and space_name not in classical_refs: alpha_val = ah_dict.get(space_name, alpha_val)
            if space_name in classical_refs: alpha_val = config.get("alpha", 0.85)
            
            x_raw, y_raw = d["Distance"].values, d[y_col].values
            x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
            
            dec_type = p_cfg.get("decimation")
            mask = np.ones(len(x_raw), dtype=bool)
            shift = 0.0
            
            if space_name not in classical_refs:
                my_group = next((g for g in groups if space_name in g), [space_name])
                group_idx = groups.index(my_group) if my_group in groups else 0
                group_size = len(my_group)
                idx_in_group = my_group.index(space_name)
                
                x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                if x_jitter > 0 and group_size > 1:
                    shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                
                stride = get_param(p_cfg.get("stride", 3), group_idx)
                thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                
                if dec_type == "interleaved":
                    mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

            lw = 2.5 if space_name in classical_refs else config["line_width"] 
            ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
            
            if show_mk: 
                ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                           s=ms, edgecolors=edge_c, 
                           linewidths=line_w, zorder=3, alpha=alpha_val)
                
        if idx == 0:
            legend_spaces = []
            if include_fci and "FCI" in subset["Active_Space"].values:
                legend_spaces.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Active_Space"].values]
            legend_spaces.extend(valid_others)
            legend_spaces.extend(active_spaces)
            
            for space_name in legend_spaces:
                c = color_map.get(space_name, "grey")
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(space_name, "-" if space_name not in classical_refs else "--") if ls_dict else ("-" if space_name not in classical_refs else "--")
                show_mk = p_cfg.get("show_markers", True) and space_name not in classical_refs
                mk = base_marker_map.get(space_name, "o") if show_mk else None
                
                label_str = space_name.replace("_", " ") if space_name not in classical_refs else space_name
                proxy = Line2D([0], [0], color=c, label=label_str, 
                               marker=mk, markersize=8, linewidth=2, linestyle=ls)
                legend_handles.append(proxy)
                    
        if panel == "energy_zoomed" and include_fci and fci_min_x is not None:
            if config.get("show_fci_min_line", False):
                ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
            if config.get("show_fci_star", False):
                ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                           s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

        ax.set_title(f"({chr(97 + idx)})", loc='center')
        ax.set_xlabel(r"Distance $d$ ($\AA$)") 
        ax.grid(False)
        
        if panel == "energy_raw":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if "ylim_energy_raw" in config: ax.set_ylim(config["ylim_energy_raw"])
        elif panel == "energy_zoomed":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_zoomed" in config: ax.set_xlim(config["xlim_energy_zoomed"])
            if "ylim_energy_zoomed" in config: ax.set_ylim(config["ylim_energy_zoomed"])
        elif panel == "time":
            ax.set_ylabel("Time (s)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if config.get("log_time", True):
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
                ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=(0.2, 0.4, 0.6, 0.8), numticks=10))
                ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        elif panel == "iterations":
            ax.set_ylabel("Iterations (nit)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
        elif panel == "evaluations":
            ax.set_ylabel("Circuit Executions") # Removed (nfev)
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])

        if panel != "time" or not config.get("log_time", True):
            y_decimals = config.get("y_decimals", {}).get(panel)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

    requested_rows = config.get("legend_rows", 1)
    calculated_ncol = max(1, int(np.ceil(len(legend_handles) / requested_rows)))
    
    fig.legend(handles=legend_handles, 
               loc=config.get("legend_loc", "lower center"), 
               bbox_to_anchor=config.get("legend_bbox", (0.5, 1.02)), 
               ncol=config.get("legend_ncol", calculated_ncol), 
               frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "show_outline": False, 
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-.", "CAS_8_8": ":"} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60, "CAS_8_8": 40} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7, "CAS_8_8": 0.6} if False else None,
        "x_jitter": 0.0,
        "groups": [["CAS_2_2", "CAS_4_4"], ["CAS_6_6", "CAS_8_8"]], 
        #"decimation": "interleaved", 
        "decimation": None, 
        "stride": [1, 2], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-.", "CAS_8_8": ":"} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60, "CAS_8_8": 40} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7, "CAS_8_8": 0.6} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    
    "time": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-.", "CAS_8_8": ":"} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60, "CAS_8_8": 40} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7, "CAS_8_8": 0.6} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 1.0, "sparse_stride": 4
    },
    
    "evaluations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-.", "CAS_8_8": ":"} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60, "CAS_8_8": 40} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7, "CAS_8_8": 0.6} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_CO2 = {
    
    #"panels": ["energy_raw", "energy_zoomed", "time", "evaluations"],
    "panels": ["energy_raw", "energy_zoomed", "evaluations"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "time": "raw",      
        "iterations": "raw",      
        "evaluations": "raw"
    },
    
    "include_fci_baseline": True, 
    "show_fci_min_line": True,    
    "show_fci_star": False,
    "include_other_baselines": ["HF", "DFT"], 
    
    "poly_degree": 4,             
    "fit_interval": (1.0, 1.4),   
    
    "global_xlim": (0.8, 2.5),
    "xlim_energy_raw": (0.6, 2.6),
    "ylim_energy_raw": (-186, -183), 
    
    "xlim_energy_zoomed": (1.0, 1.4),
    "ylim_energy_zoomed": (-185.30, -184.80), 
    
    "log_time": True,
    
    "y_decimals": {
        "energy_raw": 1,
        "energy_zoomed": 2
    },
    
    "fig_width": 18,
    "fig_height": 5,
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 1.0,           
    "outline_color": "white",      
    
    "wspace": 0.35,  
    
    "legend_rows": 1,              
    "legend_loc": "lower center",  
    "legend_bbox": (0.5, 1.02),    
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*']
}

plot_active_space_analysis(
    df_co2, 
    method_filter="aer_statevector", 
    config=PLOT_CONFIG_CO2, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    #save_image="co2_active_space_comparison_final"
)

# 2. Tapered

In [ ]:
filename_aer = "co2_tapered_mapper_comparison_final.pkl"
raw_data_aer = load_data(filename_aer) 

filename_classical = "co2_classical_benchmarks.pkl"
raw_data_classical = load_data(filename_classical)

records = []

if raw_data_aer is not None:
    for label, data_list in raw_data_aer.items():
        parts = label.split("_")
        mapper_name = parts[1]
        cas_label = f"CAS_{parts[3]}_{parts[4]}"
        
        for data_point in data_list:
            dist = data_point["distance"]
            energy = data_point["energy"]
            metrics = data_point.get("metrics", {})
            
            evals = metrics.get('cost_function_evals', metrics.get('nfev', 0))
            iterations = metrics.get('nit', 0)
            total_time = metrics.get('total_time', 0.0)
            
            records.append({
                "Distance": dist,
                "Method": mapper_name, 
                "Active_Space": cas_label,
                "Energy": energy,
                "Time": total_time,
                "Evaluations": evals,
                "Iterations": iterations
            })

if raw_data_classical is not None:
    for data_point in raw_data_classical:
        dist = data_point["distance"]
        
        if data_point.get("energy_fci") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "FCI", 
                            "Energy": data_point["energy_fci"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})
        if data_point.get("energy_hf") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "HF", 
                            "Energy": data_point["energy_hf"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})
        if data_point.get("energy_dft") is not None:
            records.append({"Distance": dist, "Method": "classical", "Active_Space": "DFT", 
                            "Energy": data_point["energy_dft"], "Time": 0.0, "Evaluations": 0, "Iterations": 0})

df_co2_tapered = pd.DataFrame(records)

In [ ]:
def plot_active_space_analysis(df, method_filter, config, visuals, use_visuals=False, save_image=False):
    subset = df[(df["Method"] == method_filter) | (df["Method"] == "classical")].copy()
    if subset.empty: return
    
    classical_refs = ["FCI", "HF", "DFT"]
    active_spaces = [a for a in subset["Active_Space"].unique() if a not in classical_refs]
    
    include_fci = config.get("include_fci_baseline", True)
    other_baselines = config.get("include_other_baselines", [])
    
    palette = sns.color_palette("tab10", n_colors=len(active_spaces))
    color_map = dict(zip(active_spaces, palette))
    
    color_map["FCI"] = "#000000"    
    color_map["HF"] = "#9467bd"     
    color_map["DFT"] = "#17becf"    
    
    base_marker_map = dict(zip(active_spaces, config["markers"][:len(active_spaces)]))
    for ref in classical_refs:
        base_marker_map[ref] = ""

    panels = config.get("panels", [])
    ncols = len(panels)
    if ncols == 0: return
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 5)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h))
    
    if ncols == 1: axes = [axes]
        
    plt.subplots_adjust(wspace=config["wspace"])

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 1.16, min(y)], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.5))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    fci_min_x, fci_min_y = None, None
    if include_fci and "FCI" in subset["Active_Space"].values:
        fci_data_df = subset[subset["Active_Space"] == "FCI"].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values
        if len(fci_dist) > 0:
            zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    legend_handles = []

    for idx, panel in enumerate(panels):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        y_col = "Energy"
        if panel == "time": y_col = "Time"
        elif panel == "iterations": y_col = "Iterations"
        elif panel == "evaluations": y_col = "Evaluations"
        
        opts_to_plot = []
        if panel in ["energy_raw", "energy_zoomed"]:
            if include_fci and "FCI" in subset["Active_Space"].values:
                opts_to_plot.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Active_Space"].values]
            opts_to_plot.extend(valid_others)
        opts_to_plot.extend(active_spaces)
        
        groups = p_cfg.get("groups", [])
        if not groups: groups = [active_spaces] 
        
        for b_idx, space_name in enumerate(opts_to_plot):
            d = subset[subset["Active_Space"] == space_name].sort_values("Distance").dropna(subset=[y_col])
            
            if panel == "time" and config.get("log_time", True):
                d = d[d[y_col] > 0]
                
            if len(d) == 0: continue
            
            c = color_map.get(space_name, "grey")
            mk = base_marker_map.get(space_name, "o")
            
            ls_dict = p_cfg.get("line_styles")
            ls = ls_dict.get(space_name, "-" if space_name not in classical_refs else "--") if ls_dict else ("-" if space_name not in classical_refs else "--")
            show_mk = p_cfg.get("show_markers", True) and space_name not in classical_refs
            
            show_outline = p_cfg.get("show_outline", False)
            edge_c = config.get("outline_color", "white") if show_outline else "none"
            line_w = config.get("outline_width", 1.0) if show_outline else 0.0
            
            ms = config.get("marker_size", 50)
            mh_dict = p_cfg.get("marker_hierarchy")
            if mh_dict is not None: ms = mh_dict.get(space_name, ms)
            if panel == "energy_zoomed": ms *= 1.5
            
            alpha_val = config.get("alpha", 0.85)
            ah_dict = p_cfg.get("alpha_hierarchy")
            if ah_dict is not None and space_name not in classical_refs: alpha_val = ah_dict.get(space_name, alpha_val)
            if space_name in classical_refs: alpha_val = config.get("alpha", 0.85)
            
            x_raw, y_raw = d["Distance"].values, d[y_col].values
            x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
            
            dec_type = p_cfg.get("decimation")
            mask = np.ones(len(x_raw), dtype=bool)
            shift = 0.0
            
            if space_name not in classical_refs:
                my_group = next((g for g in groups if space_name in g), [space_name])
                group_idx = groups.index(my_group) if my_group in groups else 0
                group_size = len(my_group)
                idx_in_group = my_group.index(space_name)
                
                x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                if x_jitter > 0 and group_size > 1:
                    shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                
                stride = get_param(p_cfg.get("stride", 3), group_idx)
                thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                
                if dec_type == "interleaved":
                    mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

            lw = 2.5 if space_name in classical_refs else config["line_width"] 
            ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
            
            if show_mk: 
                ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                           s=ms, edgecolors=edge_c, 
                           linewidths=line_w, zorder=3, alpha=alpha_val)
                
        if idx == 0:
            legend_spaces = []
            if include_fci and "FCI" in subset["Active_Space"].values:
                legend_spaces.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Active_Space"].values]
            legend_spaces.extend(valid_others)
            legend_spaces.extend(active_spaces)
            
            for space_name in legend_spaces:
                c = color_map.get(space_name, "grey")
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(space_name, "-" if space_name not in classical_refs else "--") if ls_dict else ("-" if space_name not in classical_refs else "--")
                show_mk = p_cfg.get("show_markers", True) and space_name not in classical_refs
                mk = base_marker_map.get(space_name, "o") if show_mk else None
                
                label_str = space_name.replace("_", " ") if space_name not in classical_refs else space_name
                proxy = Line2D([0], [0], color=c, label=label_str, 
                               marker=mk, markersize=8, linewidth=2, linestyle=ls)
                legend_handles.append(proxy)
                    
        if panel == "energy_zoomed" and include_fci and fci_min_x is not None:
            if config.get("show_fci_min_line", False):
                ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
            if config.get("show_fci_star", False):
                ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                           s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

        ax.set_title(f"({chr(97 + idx)})", loc='center')
        ax.set_xlabel(r"Distance $d$ ($\AA$)")
        ax.grid(False)
        
        if panel == "energy_raw":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if "ylim_energy_raw" in config: ax.set_ylim(config["ylim_energy_raw"])
        elif panel == "energy_zoomed":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_zoomed" in config: ax.set_xlim(config["xlim_energy_zoomed"])
            if "ylim_energy_zoomed" in config: ax.set_ylim(config["ylim_energy_zoomed"])
        elif panel == "time":
            ax.set_ylabel("Time (s)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if config.get("log_time", True):
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
                ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=(0.2, 0.4, 0.6, 0.8), numticks=10))
                ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        elif panel == "iterations":
            ax.set_ylabel("Iterations (nit)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
        elif panel == "evaluations":
            ax.set_ylabel("Circuit Executions")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])

        if panel != "time" or not config.get("log_time", True):
            y_decimals = config.get("y_decimals", {}).get(panel)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

    requested_rows = config.get("legend_rows", 1)
    calculated_ncol = max(1, int(np.ceil(len(legend_handles) / requested_rows)))
    
    fig.legend(handles=legend_handles, 
               loc=config.get("legend_loc", "lower center"), 
               bbox_to_anchor=config.get("legend_bbox", (0.5, 1.02)), 
               ncol=config.get("legend_ncol", calculated_ncol), 
               frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "show_outline": False, 
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-."} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["CAS_2_2", "CAS_4_4", "CAS_6_6"]], 
        "decimation": "interleaved", 
        "stride": [1], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-."} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 0.05, "sparse_stride": 4
    },
    
    "time": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-."} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None, 
        "stride": 4, "density_threshold": 1.0, "sparse_stride": 4
    },
    
    "evaluations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"CAS_2_2": "-", "CAS_4_4": "--", "CAS_6_6": "-."} if False else None,
        "marker_hierarchy": {"CAS_2_2": 120, "CAS_4_4": 90, "CAS_6_6": 60} if False else None,
        "alpha_hierarchy": {"CAS_2_2": 1.0, "CAS_4_4": 0.85, "CAS_6_6": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [],
        "decimation": None,
        "stride": 4, "density_threshold": 50.0, "sparse_stride": 4
    }
}

PLOT_CONFIG_CO2 = {
    "panels": ["energy_raw", "energy_zoomed", "time", "evaluations"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "time": "raw",      
        "iterations": "raw",      
        "evaluations": "raw"
    },
    
    "include_fci_baseline": True, 
    "show_fci_min_line": True,    
    "show_fci_star": True,
    "include_other_baselines": ["HF", "DFT"], 
    
    "poly_degree": 4,             
    "fit_interval": (1.0, 1.4),   
    
    "global_xlim": (0.8, 2.6),
    "xlim_energy_raw": (0.8, 2.6),
    "ylim_energy_raw": (-185.20, -183.80),
    
    "xlim_energy_zoomed": (1.10, 1.35),
    "ylim_energy_zoomed": (-185.15, -184.80),
    
    "log_time": True,
    
    "y_decimals": {
        "energy_raw": 1,
        "energy_zoomed": 2
    },
    
    "fig_width": 18,
    "fig_height": 5,
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 1.0,           
    "outline_color": "white",      
    
    "wspace": 0.35,  
    
    "legend_rows": 1,              
    "legend_loc": "lower center",  
    "legend_bbox": (0.5, 1.02),    
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*']
}

plot_active_space_analysis(
    df_co2_tapered, 
    method_filter="JordanWigner", 
    config=PLOT_CONFIG_CO2, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    #save_image="co2_tapered_mapper_jordanwigner_final"
)

# 3. Mapper

In [ ]:
import pandas as pd
from IPython.display import display

filename_aer = "co2_mapper_matrix_comparison_Aer_final.pkl"
raw_data_aer = load_data(filename_aer) 

filename_classical = "co2_classical_benchmarks.pkl"
raw_data_classical = load_data(filename_classical)

records = []

if raw_data_aer is not None:
    for label, data_list in raw_data_aer.items():
        parts = label.split("_")
        mapper_name = parts[0]
        cas_label = f"CAS_{parts[2]}_{parts[3]}"
        
        for data_point in data_list:
            metrics = data_point.get("metrics", {})
            records.append({
                "Distance": data_point["distance"],
                "Method": cas_label,       
                "Mapper": mapper_name,     
                "Energy": data_point["energy"],
                "Time": metrics.get('total_time', 0.0),
                "Evaluations": metrics.get('cost_function_evals', metrics.get('nfev', 0)),
                "Iterations": metrics.get('nit', 0),
                "cnot_count": metrics.get('cnot_count', 0),
                "circuit_depth": metrics.get('circuit_depth', 0),
                "num_qubits": metrics.get('num_qubits', 0)
            })

if raw_data_classical is not None:
    for data_point in raw_data_classical:
        dist = data_point["distance"]
        
        def append_classical(ref_name, e_val):
            if e_val is not None:
                records.append({
                    "Distance": dist, "Method": "classical", "Mapper": ref_name, 
                    "Energy": e_val, "Time": 0.0, "Evaluations": 0, "Iterations": 0,
                    "cnot_count": 0, "circuit_depth": 0, "num_qubits": 0
                })
                
        append_classical("FCI", data_point.get("energy_fci"))
        append_classical("HF", data_point.get("energy_hf"))
        append_classical("DFT", data_point.get("energy_dft"))

df_co2_mapper = pd.DataFrame(records)

df_quantum_only = df_co2_mapper[df_co2_mapper["Method"] != "classical"].copy()
static_summary = df_quantum_only.drop_duplicates(subset=["Method", "Mapper"])[["Method", "Mapper", "num_qubits", "circuit_depth", "cnot_count"]]
static_summary = static_summary.sort_values(by=["Method", "Mapper"]).reset_index(drop=True)
display(static_summary)

In [ ]:
def plot_mapper_analysis(df, method_filter, config, visuals, use_visuals=False, save_image=False):
    subset = df[(df["Method"] == method_filter) | (df["Method"] == "classical")].copy()
    if subset.empty: return
    
    classical_refs = ["FCI", "HF", "DFT"]
    mappers_list = [a for a in subset["Mapper"].unique() if a not in classical_refs]
    
    include_fci = config.get("include_fci_baseline", True)
    other_baselines = config.get("include_other_baselines", [])
    
    palette = sns.color_palette("tab10", n_colors=len(mappers_list))
    color_map = dict(zip(mappers_list, palette))
    
    color_map["FCI"] = "#000000"    
    color_map["HF"] = "#9467bd"     
    color_map["DFT"] = "#17becf"    
    
    base_marker_map = dict(zip(mappers_list, config["markers"][:len(mappers_list)]))
    for ref in classical_refs:
        base_marker_map[ref] = ""

    panels = config.get("panels", [])
    ncols = len(panels)
    if ncols == 0: return
    
    fig_w = config.get("fig_width", 18)
    fig_h = config.get("fig_height", 5)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, fig_h))
    
    if ncols == 1: axes = [axes]
        
    plt.subplots_adjust(wspace=config["wspace"])

    def morse_func(r, De, a, re, E_shift):
        return De * (1 - np.exp(-a * (r - re)))**2 + E_shift

    def get_plot_curve(x, y, fit_type, cfg):
        idx = np.argsort(x)
        x, y = x[idx], y[idx]
        
        if fit_type == "raw": return x, y
        x_dense = np.linspace(min(x), max(x), 500)
        
        if fit_type == "spline":
            try: 
                x_clean, unique_idx = np.unique(x, return_index=True)
                y_clean = y[unique_idx]
                if len(x_clean) > 3: return x_dense, make_interp_spline(x_clean, y_clean, k=3)(x_dense)
                return x, y
            except: return x, y
            
        elif fit_type == "morse":
            try:
                mask = x > 0.45
                popt, _ = curve_fit(morse_func, x[mask], y[mask], p0=[0.2, 1.5, 1.16, min(y)], maxfev=20000)
                return x_dense, morse_func(x_dense, *popt)
            except: return x, y
            
        elif fit_type == "poly":
            interval = cfg.get("fit_interval", (0.5, 1.5))
            mask = (x >= interval[0]) & (x <= interval[1])
            if len(x[mask]) > cfg.get("poly_degree", 4):
                coeffs = np.polyfit(x[mask], y[mask], cfg.get("poly_degree", 4))
                x_local = np.linspace(interval[0], interval[1], 300)
                return x_local, np.poly1d(coeffs)(x_local)
            return x, y
            
        return x, y 

    def get_param(param_val, g_idx):
        if isinstance(param_val, list):
            return param_val[g_idx] if g_idx < len(param_val) else param_val[-1]
        return param_val

    fci_min_x, fci_min_y = None, None
    if include_fci and "FCI" in subset["Mapper"].values:
        fci_data_df = subset[subset["Mapper"] == "FCI"].sort_values("Distance")
        fci_dist = fci_data_df["Distance"].values
        fci_energies = fci_data_df["Energy"].values
        if len(fci_dist) > 0:
            zoom_fit = config.get("fit_methods", {}).get("energy_zoomed", "spline")
            f_x, f_y = get_plot_curve(fci_dist, fci_energies, zoom_fit, config)
            f_min_idx = np.argmin(f_y)
            fci_min_x, fci_min_y = f_x[f_min_idx], f_y[f_min_idx]

    legend_handles = []

    for idx, panel in enumerate(panels):
        ax = axes[idx]
        p_cfg = visuals.get(panel, {}) if use_visuals else {}
        fit_method = config.get("fit_methods", {}).get(panel, "raw")
        
        y_col = "Energy"
        if panel == "time": y_col = "Time"
        elif panel == "iterations": y_col = "Iterations"
        elif panel == "evaluations": y_col = "Evaluations"
        
        opts_to_plot = []
        if panel in ["energy_raw", "energy_zoomed"]:
            if include_fci and "FCI" in subset["Mapper"].values:
                opts_to_plot.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Mapper"].values]
            opts_to_plot.extend(valid_others)
        opts_to_plot.extend(mappers_list)
        
        groups = p_cfg.get("groups", [])
        if not groups: groups = [mappers_list] 
        
        for b_idx, m_name in enumerate(opts_to_plot):
            d = subset[subset["Mapper"] == m_name].sort_values("Distance").dropna(subset=[y_col])
            
            if panel == "time" and config.get("log_time", True):
                d = d[d[y_col] > 0]
                
            if len(d) == 0: continue
            
            c = color_map.get(m_name, "grey")
            mk = base_marker_map.get(m_name, "o")
            
            ls_dict = p_cfg.get("line_styles")
            ls = ls_dict.get(m_name, "-" if m_name not in classical_refs else "--") if ls_dict else ("-" if m_name not in classical_refs else "--")
            show_mk = p_cfg.get("show_markers", True) and m_name not in classical_refs
            
            show_outline = p_cfg.get("show_outline", False)
            edge_c = config.get("outline_color", "white") if show_outline else "none"
            line_w = config.get("outline_width", 1.0) if show_outline else 0.0
            
            ms = config.get("marker_size", 50)
            mh_dict = p_cfg.get("marker_hierarchy")
            if mh_dict is not None: ms = mh_dict.get(m_name, ms)
            if panel == "energy_zoomed": ms *= 1.5
            
            alpha_val = config.get("alpha", 0.85)
            ah_dict = p_cfg.get("alpha_hierarchy")
            if ah_dict is not None and m_name not in classical_refs: alpha_val = ah_dict.get(m_name, alpha_val)
            if m_name in classical_refs: alpha_val = config.get("alpha", 0.85)
            
            x_raw, y_raw = d["Distance"].values, d[y_col].values
            x_curve, y_curve = get_plot_curve(x_raw, y_raw, fit_method, config)
            
            dec_type = p_cfg.get("decimation")
            mask = np.ones(len(x_raw), dtype=bool)
            shift = 0.0
            
            if m_name not in classical_refs:
                my_group = next((g for g in groups if m_name in g), [m_name])
                group_idx = groups.index(my_group) if my_group in groups else 0
                group_size = len(my_group)
                idx_in_group = my_group.index(m_name)
                
                x_jitter = get_param(p_cfg.get("x_jitter", 0.0), group_idx)
                if x_jitter > 0 and group_size > 1:
                    shift = (idx_in_group - (group_size - 1) / 2.0) * x_jitter
                
                stride = get_param(p_cfg.get("stride", 3), group_idx)
                thresh = get_param(p_cfg.get("density_threshold", 0.05), group_idx)
                sparse_stride = get_param(p_cfg.get("sparse_stride", 4), group_idx)
                
                if dec_type == "interleaved":
                    mask = (np.arange(len(x_raw)) % stride) == (idx_in_group % stride)
                elif dec_type == "density":
                    dy = np.abs(np.gradient(y_raw))
                    mask = (dy > thresh) | ((np.arange(len(x_raw)) - idx_in_group) % sparse_stride == 0)

            lw = 2.5 if m_name in classical_refs else config["line_width"] 
            ax.plot(x_curve, y_curve, color=c, linestyle=ls, lw=lw, alpha=alpha_val, zorder=2)
            
            if show_mk: 
                ax.scatter(x_raw[mask] + shift, y_raw[mask], color=c, marker=mk, 
                           s=ms, edgecolors=edge_c, 
                           linewidths=line_w, zorder=3, alpha=alpha_val)
                
        if idx == 0:
            legend_spaces = []
            if include_fci and "FCI" in subset["Mapper"].values:
                legend_spaces.append("FCI")
            valid_others = [b for b in other_baselines if b in subset["Mapper"].values]
            legend_spaces.extend(valid_others)
            legend_spaces.extend(mappers_list)
            
            for m_name in legend_spaces:
                c = color_map.get(m_name, "grey")
                ls_dict = p_cfg.get("line_styles")
                ls = ls_dict.get(m_name, "-" if m_name not in classical_refs else "--") if ls_dict else ("-" if m_name not in classical_refs else "--")
                show_mk = p_cfg.get("show_markers", True) and m_name not in classical_refs
                mk = base_marker_map.get(m_name, "o") if show_mk else None
                
                label_str = m_name.replace("_", " ") if m_name not in classical_refs else m_name
                proxy = Line2D([0], [0], color=c, label=label_str, 
                               marker=mk, markersize=8, linewidth=2, linestyle=ls)
                legend_handles.append(proxy)
                    
        if panel == "energy_zoomed" and include_fci and fci_min_x is not None:
            if config.get("show_fci_min_line", False):
                ax.axvline(x=fci_min_x, color='black', linestyle=':', alpha=0.5, zorder=1)
            if config.get("show_fci_star", False):
                ax.scatter(fci_min_x, fci_min_y, color='red', marker='*', 
                           s=config.get("marker_size", 50) * 6, edgecolors='none', zorder=10)

        ax.set_title(f"({chr(97 + idx)})", loc='center')
        ax.set_xlabel(r"Distance $d$ ($\AA$)")
        ax.grid(False)
        
        if panel == "energy_raw":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if "ylim_energy_raw" in config: ax.set_ylim(config["ylim_energy_raw"])
        elif panel == "energy_zoomed":
            ax.set_ylabel("Energy (Ha)")
            if "xlim_energy_zoomed" in config: ax.set_xlim(config["xlim_energy_zoomed"])
            if "ylim_energy_zoomed" in config: ax.set_ylim(config["ylim_energy_zoomed"])
        elif panel == "time":
            ax.set_ylabel("Time (s)")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
            if config.get("log_time", True):
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
                ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=(0.2, 0.4, 0.6, 0.8), numticks=10))
                ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        elif panel == "iterations":
            ax.set_ylabel("Iterations")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])
        elif panel == "evaluations":
            ax.set_ylabel("Circuit Executions")
            if "xlim_energy_raw" in config: ax.set_xlim(config["xlim_energy_raw"])

        if panel != "time" or not config.get("log_time", True):
            y_decimals = config.get("y_decimals", {}).get(panel)
            if y_decimals is not None:
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{y_decimals}f'))
                
            x_decimals = config.get("x_decimals", {}).get(panel)
            if x_decimals is not None:
                ax.xaxis.set_major_formatter(ticker.FormatStrFormatter(f'%.{x_decimals}f'))

    requested_rows = config.get("legend_rows", 1)
    calculated_ncol = max(1, int(np.ceil(len(legend_handles) / requested_rows)))
    
    fig.legend(handles=legend_handles, 
               loc=config.get("legend_loc", "lower center"), 
               bbox_to_anchor=config.get("legend_bbox", (0.5, 1.02)), 
               ncol=config.get("legend_ncol", calculated_ncol), 
               frameon=True, fancybox=True, edgecolor='black')

    if save_image:
        save_png(save_image, fig)
        save_pdf(save_image, fig)

    plt.show()

In [ ]:
ENABLE_ADVANCED_VISUALS = True

VISUAL_ENHANCEMENTS = {
    "energy_raw": {
        "show_markers": True,
        "show_outline": False, 
        "line_styles": {"JordanWigner": "-", "BravyiKitaev": "--", "Parity": "-."} if False else None,
        "marker_hierarchy": {"JordanWigner": 120, "BravyiKitaev": 90, "Parity": 60} if False else None,
        "alpha_hierarchy": {"JordanWigner": 1.0, "BravyiKitaev": 0.85, "Parity": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["JordanWigner", "BravyiKitaev", "Parity"]], 
        "decimation": "interleaved", 
        "stride": [3], 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "energy_zoomed": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"JordanWigner": "-", "BravyiKitaev": "--", "Parity": "-."} if False else None,
        "marker_hierarchy": {"JordanWigner": 120, "BravyiKitaev": 90, "Parity": 60} if False else None,
        "alpha_hierarchy": {"JordanWigner": 1.0, "BravyiKitaev": 0.85, "Parity": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["JordanWigner", "BravyiKitaev", "Parity"]],
        "decimation": None,
        "stride": 4, 
        "density_threshold": 0.05, 
        "sparse_stride": 4
    },
    
    "time": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"JordanWigner": "-", "BravyiKitaev": "--", "Parity": "-."} if False else None,
        "marker_hierarchy": {"JordanWigner": 120, "BravyiKitaev": 90, "Parity": 60} if False else None,
        "alpha_hierarchy": {"JordanWigner": 1.0, "BravyiKitaev": 0.85, "Parity": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["JordanWigner", "BravyiKitaev", "Parity"]],
        "decimation": None, 
        "stride": 4, 
        "density_threshold": 1.0, 
        "sparse_stride": 4
    },
    
    "evaluations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"JordanWigner": "-", "BravyiKitaev": "--", "Parity": "-."} if False else None,
        "marker_hierarchy": {"JordanWigner": 120, "BravyiKitaev": 90, "Parity": 60} if False else None,
        "alpha_hierarchy": {"JordanWigner": 1.0, "BravyiKitaev": 0.85, "Parity": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["JordanWigner", "BravyiKitaev", "Parity"]],
        "decimation": None,
        "stride": 4, 
        "density_threshold": 50.0, 
        "sparse_stride": 4
    },
    
    "iterations": {
        "show_markers": True,
        "show_outline": False,
        "line_styles": {"JordanWigner": "-", "BravyiKitaev": "--", "Parity": "-."} if False else None,
        "marker_hierarchy": {"JordanWigner": 120, "BravyiKitaev": 90, "Parity": 60} if False else None,
        "alpha_hierarchy": {"JordanWigner": 1.0, "BravyiKitaev": 0.85, "Parity": 0.7} if False else None,
        "x_jitter": 0.0,
        "groups": [["JordanWigner", "BravyiKitaev", "Parity"]],
        "decimation": None,
        "stride": 4, 
        "density_threshold": 10.0, 
        "sparse_stride": 4
    }
}

PLOT_CONFIG_CO2 = {
    
    #"panels": ["energy_raw", "energy_zoomed", "time", "evaluations"],
    "panels": ["energy_raw", "energy_zoomed", "evaluations"],
    
    "fit_methods": {
        "energy_raw": "morse",      
        "energy_zoomed": "morse",  
        "time": "raw",      
        "iterations": "raw",      
        "evaluations": "raw"
    },
    
    "include_fci_baseline": True, 
    "show_fci_min_line": True,    
    "show_fci_star": True,
    #"include_other_baselines": ["HF", "DFT"], 
    "include_other_baselines": [], 
    
    "poly_degree": 4,             
    "fit_interval": (1.0, 1.4),   
    
    "global_xlim": (0.8, 2.5),
    "xlim_energy_raw": (0.8, 2.6),
    "ylim_energy_raw": (-185.5, -183), 
    
    "xlim_energy_zoomed": (1.0, 1.4),
    "ylim_energy_zoomed": (-185.30, -184.80), 
    #"ylim_energy_zoomed": (-185.15, -185.10),
    
    "log_time": True,
    
    "y_decimals": {
        "energy_raw": 1,
        "energy_zoomed": 2
    },
    
    "fig_width": 18,
    "fig_height": 5,
    
    "line_width": 1.5,
    "marker_size": 60,            
    "alpha": 0.85,                
    "outline_width": 1.0,           
    "outline_color": "white",      
    
    "wspace": 0.35,  
    
    "legend_rows": 1,              
    "legend_loc": "lower center",  
    "legend_bbox": (0.5, 1.02),    
    "markers": ['o', 's', '^', 'D', 'X', 'P', 'v', '*']
}

plot_mapper_analysis(
    df_co2_mapper, 
    method_filter="CAS_4_4", 
    config=PLOT_CONFIG_CO2, 
    visuals=VISUAL_ENHANCEMENTS,
    use_visuals=ENABLE_ADVANCED_VISUALS,
    #save_image="co2_mapper_comparison_cas66_final"
)